In [1]:
from pathlib import Path
import numpy as np
import torch
from typing import List
from torch.nn.utils.rnn import pad_sequence
from mltrainer import rnn_models, Trainer
from torch import optim

from mads_datasets import datatools
import mltrainer
mltrainer.__version__

'0.2.5'

# 1 Iterators
We will be using an interesting dataset. [link](https://tev.fbk.eu/resources/smartwatch)

From the site:
> The SmartWatch Gestures Dataset has been collected to evaluate several gesture recognition algorithms for interacting with mobile applications using arm gestures. Eight different users performed twenty repetitions of twenty different gestures, for a total of 3200 sequences. Each sequence contains acceleration data from the 3-axis accelerometer of a first generation Sony SmartWatch™, as well as timestamps from the different clock sources available on an Android device. The smartwatch was worn on the user's right wrist. 


In [2]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor
preprocessor = PaddedPreprocessor()

gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
streamers = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]

2026-03-06 13:16:08.398 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/jgerrits/.cache/mads_datasets/gestures
100%|██████████| 651/651 [00:00<00:00, 4314.85it/s]


In [3]:
len(train), len(valid)

(81, 20)

In [4]:
trainstreamer = train.stream()
validstreamer = valid.stream()
x, y = next(iter(trainstreamer))
x.shape, y

(torch.Size([32, 30, 3]),
 tensor([ 9, 12,  3, 12, 16,  3, 16, 18, 12, 13,  0,  3,  8, 19,  5, 10,  2,  4,
          1,  9,  6,  8,  2, 17, 18,  5, 19,  4,  4,  1, 10,  8]))

Can you make sense of the shape?
What does it mean that the shapes are sometimes (32, 27, 3), but a second time might look like (32, 30, 3)? In other words, the second (or first, if you insist on starting at 0) dimension changes. Why is that? How does the model handle this? Do you think this is already padded, or still has to be padded?


Answer: The shape of the data is batch size (32), timestamps (26) and features per timestamp (2).
The difference between 26 and 30 is the different amount of timesteps for different gestures.
The tensor consists of all the output classes (32 outputs), what makes this a sequence-to-one classification problem.

I think the model can handle the different amount of timesteps, because it is a recurrent process. In the final steps of the network, the number of outputs is brought back to the needed number of 20 outputs. 

Discussed with Sam in class: We don't see why the data is padded up to now?
Padded-Preprocesser is added, but not configurated yet?

# 2 Excercises
Lets test a basemodel, and try to improve upon that.

Fill the gestures.gin file with relevant settings for `input_size`, `hidden_size`, `num_layers` and `horizon` (which, in our case, will be the number of classes...)

As a rule of thumbs: start lower than you expect to need!

In [5]:
from mltrainer import TrainerSettings, ReportTypes
from mltrainer.metrics import Accuracy

accuracy = Accuracy()


In [6]:
model = rnn_models.BaseRNN(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    horizon=20,
)

Test the model. What is the output shape you need? Remember, we are doing classification!

In [7]:
yhat = model(x)
yhat.shape

torch.Size([32, 20])

Test the accuracy

In [8]:
accuracy(y, yhat)

0.09375

What do you think of the accuracy? What would you expect from blind guessing?

Check shape of `y` and `yhat`

In [9]:
yhat.shape, y.shape

(torch.Size([32, 20]), torch.Size([32]))

And look at the output of yhat

In [10]:
yhat[0]

tensor([ 0.1486, -0.0787,  0.0398,  0.0311,  0.2472,  0.1004, -0.1272,  0.0793,
         0.0287,  0.0145,  0.0759,  0.1250,  0.0450,  0.1562, -0.0351, -0.1702,
        -0.0168,  0.2388,  0.0674, -0.0425], grad_fn=<SelectBackward0>)

Does this make sense to you? If you are unclear, go back to the classification problem with the MNIST, where we had 10 classes.

We have a classification problem, so we need Cross Entropy Loss.
Remember, [this has a softmax built in](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) 

In [11]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(yhat, y)
loss

tensor(2.9916, grad_fn=<NllLossBackward0>)

In [12]:
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

# on my mac, at least for the BaseRNN model, mps does not speed up training
# probably because the overhead of copying the data to the GPU is too high
# so i override the device to cpu
device = "cpu"
# however, it might speed up training for larger models, with more parameters

using cpu


Set up the settings for the trainer and the different types of logging you want

In [13]:
import mlflow
from datetime import datetime

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)


2026/03/06 13:16:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/06 13:16:09 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


In [14]:
print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: sqlite:///mlflow.db


# Loop starts here

In [27]:
settings = TrainerSettings(
    epochs=10, # increase this to about 100 for training
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs = {
        "save": False, # save every best model, and restore the best one
        "verbose": True,
        "patience": 5, # number of epochs with no improvement after which training will be stopped
        "delta": 0.0, # minimum change to be considered an improvement
    }
)
settings

epochs: 10
metrics: [Accuracy]
logdir: gestures
train_steps: 81
valid_steps: 20
reporttypes: [<ReportTypes.TOML: 'TOML'>, <ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.MLFLOW: 'MLFLOW'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 5, 'delta': 0.0}

In [28]:
import torch.nn as nn
import torch
from torch import Tensor
from dataclasses import dataclass

@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

In [29]:
config = ModelConfig(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    output_size=20,
    dropout=0.0,
)


In [30]:
config = ModelConfig(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    output_size=20,
    dropout=0.1,
)

model_type = "GRU"

run_name = (
    f"{model_type}_"
    f"h{config.hidden_size}_"
    f"l{config.num_layers}_"
    f"do{config.dropout}_"
    f"ep{settings.epochs}"
)

with mlflow.start_run(run_name=run_name):

    mlflow.set_tag("model_type", model_type)
    mlflow.set_tag("developer", "your-name-here")

    mlflow.log_params({
        "input_size": config.input_size,
        "hidden_size": config.hidden_size,
        "num_layers": config.num_layers,
        "dropout": config.dropout,
        "output_size": config.output_size,
        "epochs": settings.epochs,
        "batchsize": 32,
        "optimizer": "Adam",
    })

    model = GRUmodel(config=config)

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=trainstreamer,
        validdataloader=validstreamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )

    trainer.loop()

    if not settings.earlystop_kwargs["save"]:
        tag = datetime.now().strftime("%Y%m%d-%H%M-")
        modelpath = modeldir / (tag + "model.pt")
        torch.save(model, modelpath)

2026-03-06 13:54:54.416 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260306-135454
2026-03-06 13:54:54.417 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:00<00:00, 91.67it/s]
2026-03-06 13:54:55.429 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.9152 test 2.5907 metric ['0.1000']
100%|██████████| 81/81 [00:00<00:00, 86.89it/s]
2026-03-06 13:54:56.500 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.3587 test 2.2434 metric ['0.1859']
100%|██████████| 81/81 [00:00<00:00, 86.32it/s]
2026-03-06 13:54:57.567 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 2.1408 test 2.0722 metric ['0.2281']
100%|██████████| 81/81 [00:00<00:00, 89.43it/s]
2026-03-06 13:54:58.609 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 1.9216 test 1.8326 metric ['0.3094']
100%|██████████| 81/81 [00:00<00:00, 91.73it/s]
2026-03-06 13:54:59.61

Try to update the code above by changing the hyperparameters.
    
To discern between the changes, also modify the tag mlflow.set_tag("model", "new-tag-here") where you add
a new tag of your choice. This way you can keep the models apart.

In [31]:
from pathlib import Path
print(Path("mlflow.db").resolve())

/home/jgerrits/portfolio-S3/3-hypertuning-rnn/mlflow.db


cd /home/jgerrits/portfolio-S3/3-hypertuning-rnn


mlflow ui --backend-store-uri sqlite:///mlflow.db


In [32]:
#trainer.loop() # if you want to pick up training, loop will continue from the last epoch

In [33]:
#mlflow.end_run()